# Train a rock, paper, scissors detector with LibreYOLO

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.

Install LibreYOLO.

In [ ]:
%pip install -q "libreyolo[onnx]" pillow-heif

Download today's dataset: the photos we took and labelled this morning, from Hugging Face.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("Xuban/rock-paper-scissors-mondragon", repo_type="dataset", local_dir="dataset")

Train. The model already knows how to see from COCO, and now it learns rock, paper and scissors.
This takes about 10 minutes.

In [ ]:
from libreyolo import LibreYOLO

model = LibreYOLO("LibreYOLO9t.pt")
results = model.train(data="dataset/data.yaml", epochs=25, imgsz=640)

Score the best model on the validation photos, which it never trained on. mAP50 goes from 0 to 1: higher is better.

In [ ]:
best = LibreYOLO(results["best_checkpoint"])
metrics = best.val(data="dataset/data.yaml")
print("mAP50:", round(metrics["metrics/mAP50"], 3))

Look at what it finds on a few validation photos.

In [ ]:
import glob

for photo in sorted(glob.glob("dataset/images/val/*"))[:4]:
    display(best(photo).plot(pil=True))

Download your model: best.pt for Python, best.onnx to run it anywhere.

In [ ]:
from google.colab import files

files.download(results["best_checkpoint"])
files.download(best.export(format="onnx"))

Take a photo of your hand and upload it to see what your model finds. iPhone photos work too.

In [ ]:
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()
for name in files.upload():
    display(best(ImageOps.exif_transpose(Image.open(name)).convert("RGB")).plot(pil=True))

The Roboflow photos and labels come from the [Rock Paper Scissors dataset](https://universe.roboflow.com/rock-paper-scissors-2/rock-paper-scissors-bwev7) on Roboflow Universe, CC BY 4.0.